In [13]:
import umap.umap_ as umap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import random
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
import scipy.stats as stats
import math
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [14]:
data = pd.read_csv('edited_data.csv')
data[data < 0] = 0
data["date"] = pd.to_datetime(data["Date"], format="%Y%m%d")
data["Month"] = data["date"].dt.month
data['TRAFFIC'] = data['TRAFFIC'].map(
    lambda x: math.log(x) if (pd.notnull(x) and x > 0) else x
)
# data['Vehicles'] = data['Vehicles'].map(
#     lambda x: math.log(x) if (pd.notnull(x) and x > 0) else x
# )
# data['SNOW_DAY_SUM'] = data['SNOW_DAY_SUM'].map(
#     lambda x: math.log(x) if (pd.notnull(x) and x > 0) else x
# )
data.head()

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek,date,Month
0,0,20140101,1,0,4.624973,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
1,1,20140101,0,0,4.875197,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
2,2,20140101,1,1,4.510860,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
3,3,20140101,0,1,5.087596,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
4,4,20140101,1,2,4.454347,0.12,1.9,13.0,1.9,1.0,26.0,0.0,0.0,3,2014-01-01,1


In [15]:
Month_codots = {
        1: 3, 2:1, 3:2, 4:4, 12: 5, 5:6, 11: 7, 6: 8, 7: 12, 8: 10, 9: 11, 10: 9
    }
k = {1,2,3,4,5,6}
data['WeightedMonth'] = data['Month'].map(lambda x: Month_codots[x])
data_with_snow = data[data['SNOW'] > 0]
data_with_no_snow = data[data['SNOW'] == 0]
data_with_no_snow.head()

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek,date,Month,WeightedMonth
98,98,20140103,1,0,4.804021,0.0,0.0,12.0,2.4,0.0,0.0,0.0,0.0,5,2014-01-03,1,3
99,99,20140103,0,0,5.093750,0.0,0.0,12.0,2.4,0.0,0.0,0.0,0.0,5,2014-01-03,1,3
100,100,20140103,1,1,4.382027,0.0,0.0,12.0,2.4,0.0,0.0,0.0,0.0,5,2014-01-03,1,3
101,101,20140103,0,1,4.682131,0.0,0.0,12.0,2.4,0.0,0.0,0.0,0.0,5,2014-01-03,1,3
102,102,20140103,1,2,4.219508,0.0,0.0,12.0,2.4,0.0,0.0,0.0,0.0,5,2014-01-03,1,3


In [16]:
data_with_no_snow = data_with_no_snow[data_with_no_snow['WeightedMonth'].isin(k)]
data_with_snow["Time"] = data_with_snow["Time"].astype("category")
data_with_snow["Month"] = data_with_snow["Month"].astype("category")
data_with_snow["DayOfWeek"] = data_with_snow["DayOfWeek"].astype("category")
data_with_no_snow["Time"] = data_with_no_snow["Time"].astype("category")
data_with_no_snow["Month"] = data_with_no_snow["Month"].astype("category")
data_with_no_snow["DayOfWeek"] = data_with_no_snow["DayOfWeek"].astype("category")
data_with_snow.head()

C:\Users\wsach\AppData\Local\Temp\ipykernel_28904\897346955.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_with_snow["Time"] = data_with_snow["Time"].astype("category")
C:\Users\wsach\AppData\Local\Temp\ipykernel_28904\897346955.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_with_snow["Month"] = data_with_snow["Month"].astype("category")
C:\Users\wsach\AppData\Local\Temp\ipykernel_28904\897346955.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek,date,Month,WeightedMonth
0,0,20140101,1,0,4.624973,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1,3
1,1,20140101,0,0,4.875197,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1,3
2,2,20140101,1,1,4.510860,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1,3
3,3,20140101,0,1,5.087596,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1,3
4,4,20140101,1,2,4.454347,0.12,1.9,13.0,1.9,1.0,26.0,0.0,0.0,3,2014-01-01,1,3


In [17]:
ols_snow = smf.ols(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_snow
).fit()

print(ols_snow.summary())

                            OLS Regression Results                            
Dep. Variable:                TRAFFIC   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     3929.
Date:                Fri, 05 Dec 2025   Prob (F-statistic):               0.00
Time:                        10:15:08   Log-Likelihood:                -24596.
No. Observations:               36324   AIC:                         4.929e+04
Df Residuals:                   36274   BIC:                         4.972e+04
Df Model:                          49                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [18]:
ols_no_snow = smf.ols(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_no_snow
).fit()

print(ols_no_snow.summary())

                            OLS Regression Results                            
Dep. Variable:                TRAFFIC   R-squared:                       0.883
Model:                            OLS   Adj. R-squared:                  0.883
Method:                 Least Squares   F-statistic:                     8899.
Date:                Fri, 05 Dec 2025   Prob (F-statistic):               0.00
Time:                        10:15:09   Log-Likelihood:                -25535.
No. Observations:               51940   AIC:                         5.116e+04
Df Residuals:                   51895   BIC:                         5.156e+04
Df Model:                          44                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [22]:
data_with_snow['weights'] = 1 / data_with_snow['TRAFFIC']
data_with_snow['weights'] = data_with_snow['weights'].clip(upper=50)
wls = smf.wls(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_snow,
    weights=data_with_snow['weights']
).fit()

print(wls.summary())

C:\Users\wsach\AppData\Local\Temp\ipykernel_28904\3132579671.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_with_snow['weights'] = 1 / data_with_snow['TRAFFIC']
C:\Users\wsach\AppData\Local\Temp\ipykernel_28904\3132579671.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_with_snow['weights'] = data_with_snow['weights'].clip(upper=50)


                            WLS Regression Results                            
Dep. Variable:                TRAFFIC   R-squared:                       0.667
Model:                            WLS   Adj. R-squared:                  0.667
Method:                 Least Squares   F-statistic:                     1485.
Date:                Fri, 05 Dec 2025   Prob (F-statistic):               0.00
Time:                        10:23:18   Log-Likelihood:                -72141.
No. Observations:               36324   AIC:                         1.444e+05
Df Residuals:                   36274   BIC:                         1.448e+05
Df Model:                          49                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [24]:
glm_log = smf.glm(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_snow,
    family=sm.families.Gaussian(sm.families.links.log())
).fit()

print(glm_log.summary())

C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


                 Generalized Linear Model Regression Results                  
Dep. Variable:                TRAFFIC   No. Observations:                36324
Model:                            GLM   Df Residuals:                    36274
Model Family:                Gaussian   Df Model:                           49
Link Function:                    log   Scale:                         0.22677
Method:                          IRLS   Log-Likelihood:                -24568.
Date:                Fri, 05 Dec 2025   Deviance:                       8226.0
Time:                        10:24:23   Pearson chi2:                 8.23e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.9951
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [26]:
glm_gamma = smf.glm(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_snow,
    family=sm.families.Gamma()
).fit()

print(glm_gamma.summary())

C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


                 Generalized Linear Model Regression Results                  
Dep. Variable:                TRAFFIC   No. Observations:                36324
Model:                            GLM   Df Residuals:                    36274
Model Family:                   Gamma   Df Model:                           49
Link Function:           InversePower   Scale:                       0.0062314
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Fri, 05 Dec 2025   Deviance:                       2376.6
Time:                        10:27:51   Pearson chi2:                     226.
No. Iterations:                     7   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\genmod\families\family.py:812: RuntimeWarning: divide by zero encountered in log
  ll_obs -= special.gammaln(weight_scale) + np.log(endog)
C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\genmod\generalized_linear_model.py:1891: RuntimeWarning: invalid value encountered in scalar subtract
  prsq = 1 - np.exp((self.llnull - self.llf) * (2 / self.nobs))


In [29]:
glm_nb = smf.glm(
    formula="TRAFFIC ~ C(Time) + C(WeightedMonth) + C(DayOfWeek) + SNOW_DAY_SUM + C(Condition_Code) + C(Direction) + SNOW",
    data=data_with_snow,
    family=sm.families.NegativeBinomial()
).fit()

print(glm_nb.summary())

C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:                TRAFFIC   No. Observations:                36324
Model:                            GLM   Df Residuals:                    36274
Model Family:        NegativeBinomial   Df Model:                           49
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0562e+05
Date:                Fri, 05 Dec 2025   Deviance:                       286.13
Time:                        10:29:12   Pearson chi2:                     191.
No. Iterations:                     5   Pseudo R-squ. (CS):            0.02785
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [30]:
from sklearn.linear_model import LassoCV

X = pd.get_dummies(data_with_snow.drop(columns=['TRAFFIC']), drop_first=True)
y = data_with_snow['TRAFFIC']

lasso = LassoCV(cv=5).fit(X, y)

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy

In [31]:
model = sm.tsa.SARIMAX(
    data_with_snow.TRAFFIC,
    exog=data_with_snow[['SNOW', 'SNOW_DAY_SUM']],
    order=(1,0,1),
    seasonal_order=(1,0,1,24)
).fit()

C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\wsach\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge